# 🧠 Causal Inference with CausalNex: Demand, Price & Income

## What We're Trying to Answer

> **"What actually *causes* a customer to buy — and if we intervene (e.g., lower prices, target higher-income segments), what would actually happen to demand?"**

This notebook uses the **UCI Online Retail dataset** — a real transactional dataset from a UK-based e-commerce company — and applies **CausalNex** (a Bayesian Network causal inference library) to understand the *causal structure* behind purchasing behaviour.

### Why Not Just Use Machine Learning?
A traditional ML model tells you **correlations** — e.g., *customers who got discounts bought more*. But this is misleading:
- Discounts are targeted at customers already likely to buy
- High sales periods also have high marketing spend (seasonality confounds both)
- ML cannot answer: *"If I **force** price down by 15% across all segments, what happens to demand?"*

Causal inference can. It separates *observation* from *intervention*.

### The Causal Question Hierarchy
| Level | Question | Tool |
|---|---|---|
| Association | "Do customers who buy premium products earn more?" | ML / Stats |
| Intervention | "If we lower price by 10%, what happens to demand?" | Causal Inference |
| Counterfactual | "Would this customer have bought if we hadn't sent the coupon?" | Advanced CI |

**We will operate at the Intervention level.**

---
## Step 1: Install and Import Libraries

CausalNex is not pre-installed on Kaggle. We install it along with a few supporting libraries.
- `causalnex` — the main causal inference library (Bayesian Networks + structure learning)
- `networkx` — for graph operations
- `matplotlib` / `seaborn` — for visualisation

In [ ]:
# Install CausalNex (takes ~1-2 minutes on Kaggle)
!pip install causalnex --quiet

print("✅ CausalNex installed successfully")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# CausalNex core components
from causalnex.structure import StructureModel
from causalnex.structure.notears import from_pandas
from causalnex.network import BayesianNetwork
from causalnex.discretiser import Discretiser
from causalnex.inference import InferenceEngine

print("✅ All libraries imported successfully")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   networkx: {nx.__version__}")

---
## Step 2: Load the Dataset

We use the **UCI Online Retail Dataset** — real UK e-commerce transactions from 2010–2011.

**Key columns for our causal question:**
| Column | Meaning | Role in Causal Graph |
|---|---|---|
| `UnitPrice` | Price of one unit | Potential cause of demand |
| `Quantity` | Units purchased | Proxy for demand |
| `CustomerID` | Unique customer | Used to build customer-level features |
| `Country` | Customer location | Proxy for income/market segment |
| `InvoiceDate` | Transaction date | Used to extract seasonality |
| `Description` | Product name | Used to create product tiers |

We will engineer a few additional features: `income_proxy`, `price_tier`, `season`, and our outcome: `high_demand`.

In [ ]:
# Load directly from UCI ML Repository (public, no login needed)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

print("⏳ Downloading UCI Online Retail dataset (this may take ~30 seconds)...")
df_raw = pd.read_excel(url)
print(f"✅ Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print()
print("--- First 5 rows ---")
print(df_raw.head())
print()
print("--- Column Info ---")
print(df_raw.dtypes)
print()
print("--- Missing Values ---")
print(df_raw.isnull().sum())

---
## Step 3: Data Cleaning

Before we can build any causal model, we need clean data. We will:
1. Remove cancelled orders (InvoiceNo starting with 'C')
2. Remove rows with missing CustomerID (we need customer-level features)
3. Remove rows with zero or negative prices/quantities (data errors)
4. Keep only the UK for simplicity (largest segment, reduces country noise)

**Why does data quality matter more for causal inference than ML?**  
In ML, a few bad rows just add noise. In causal inference, bad rows can *flip the direction of a causal edge* — causing the model to conclude that high demand causes low prices, when really it's the other way around.

In [ ]:
df = df_raw.copy()

print(f"📊 Starting rows: {len(df):,}")

# Step 3a: Remove cancelled orders
cancelled_mask = df['InvoiceNo'].astype(str).str.startswith('C')
df = df[~cancelled_mask]
print(f"   After removing cancellations: {len(df):,} rows (removed {cancelled_mask.sum():,})")

# Step 3b: Remove missing CustomerID
before = len(df)
df = df.dropna(subset=['CustomerID'])
print(f"   After removing missing CustomerID: {len(df):,} rows (removed {before - len(df):,})")

# Step 3c: Remove bad prices and quantities
before = len(df)
df = df[(df['UnitPrice'] > 0) & (df['Quantity'] > 0)]
print(f"   After removing zero/negative prices & quantities: {len(df):,} rows (removed {before - len(df):,})")

# Step 3d: Keep UK only
before = len(df)
df = df[df['Country'] == 'United Kingdom']
print(f"   After keeping UK only: {len(df):,} rows (removed {before - len(df):,})")

# Step 3e: Parse dates
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print()
print(f"✅ Clean dataset: {len(df):,} rows")
print()
print("--- Price distribution ---")
print(df['UnitPrice'].describe())
print()
print("--- Quantity distribution ---")
print(df['Quantity'].describe())

---
## Step 4: Feature Engineering

CausalNex works with **discrete/categorical variables** (it builds a Bayesian Network, which models conditional probability tables). So we need to:

1. Create a **customer-level** dataset (aggregate transactions per customer)
2. Engineer causal variables:
   - `income_proxy` — total spend per customer (high spenders = higher income segment)
   - `price_sensitivity` — average price paid (do they buy cheap or expensive items?)
   - `purchase_frequency` — how often they buy
   - `season` — what season most of their orders fell in
   - `high_demand` — our **outcome variable** (did this customer buy in high volume?)
3. **Discretise** all continuous variables into bins

**Think of it like this:** We're building a profile per customer that captures their economic behaviour, and then asking: *which factors in that profile causally drive high demand?*

In [ ]:
# Step 4a: Build customer-level features
print("⚙️  Building customer-level feature table...")

df['TotalSpend'] = df['UnitPrice'] * df['Quantity']
df['Month'] = df['InvoiceDate'].dt.month

# Season mapping
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

df['Season'] = df['Month'].apply(get_season)

# Most frequent season per customer
customer_season = df.groupby('CustomerID')['Season'].agg(
    lambda x: x.value_counts().index[0]
).rename('dominant_season')

# Core aggregations per customer
customer_df = df.groupby('CustomerID').agg(
    total_spend    = ('TotalSpend', 'sum'),
    avg_price      = ('UnitPrice', 'mean'),
    total_quantity = ('Quantity', 'sum'),
    num_orders     = ('InvoiceNo', 'nunique'),
    num_products   = ('StockCode', 'nunique'),
).reset_index()

customer_df = customer_df.merge(customer_season, on='CustomerID')

print(f"✅ Customer-level dataset: {len(customer_df):,} unique customers")
print()
print("--- Customer feature summary ---")
print(customer_df.describe())
print()
print("--- Season distribution ---")
print(customer_df['dominant_season'].value_counts())

In [ ]:
# Step 4b: Cap outliers (top 1% are outliers that distort causal learning)
print("⚙️  Capping outliers at 99th percentile...")

for col in ['total_spend', 'avg_price', 'total_quantity', 'num_orders', 'num_products']:
    cap = customer_df[col].quantile(0.99)
    before_max = customer_df[col].max()
    customer_df[col] = customer_df[col].clip(upper=cap)
    print(f"   {col}: max {before_max:.1f} → capped at {cap:.1f}")

print()

In [ ]:
# Step 4c: Discretise continuous variables into meaningful bins
# CausalNex Bayesian Networks require discrete variables
# We use quantile-based binning so each bin has roughly equal population
# Note: rank(method='first') is used to break ties when many customers share
# the same value (e.g., many have exactly 1 order), which would otherwise
# cause pd.qcut to fail with "Bin edges must be unique" errors.

print("⚙️  Discretising continuous variables into categorical bins...")
print()

causal_df = customer_df.copy()

# Income proxy: total spend → Low / Medium / High
causal_df['income_segment'] = pd.qcut(
    causal_df['total_spend'], q=3,
    labels=['Low_Income', 'Mid_Income', 'High_Income']
).astype(str)
print("income_segment distribution:")
print(causal_df['income_segment'].value_counts())
print()

# Price tier: avg price paid → Budget / Mid / Premium
causal_df['price_tier'] = pd.qcut(
    causal_df['avg_price'], q=3,
    labels=['Budget', 'Mid_Range', 'Premium']
).astype(str)
print("price_tier distribution:")
print(causal_df['price_tier'].value_counts())
print()

# Purchase frequency: num_orders → Low / Medium / High frequency
# Using rank(method='first') to handle ties (many customers have exactly 1 order)
causal_df['purchase_freq'] = pd.qcut(
    causal_df['num_orders'].rank(method='first'), q=3,
    labels=['Infrequent', 'Regular', 'Frequent']
).astype(str)
print("purchase_freq distribution:")
print(causal_df['purchase_freq'].value_counts())
print()

# Product variety: num_products → Narrow / Broad basket
# Using rank(method='first') to handle ties
causal_df['basket_variety'] = pd.qcut(
    causal_df['num_products'].rank(method='first'), q=2,
    labels=['Narrow_Basket', 'Broad_Basket']
).astype(str)
print("basket_variety distribution:")
print(causal_df['basket_variety'].value_counts())
print()

# OUTCOME: high_demand — did this customer buy in high volume?
quantity_threshold = causal_df['total_quantity'].median()
causal_df['high_demand'] = (causal_df['total_quantity'] > quantity_threshold).astype(int)
causal_df['high_demand'] = causal_df['high_demand'].map({0: 'Low_Demand', 1: 'High_Demand'})
print(f"high_demand (outcome) — threshold = {quantity_threshold:.0f} units:")
print(causal_df['high_demand'].value_counts())
print()

# Season stays as-is (already categorical)
print("dominant_season distribution:")
print(causal_df['dominant_season'].value_counts())

In [ ]:
# Step 4d: Build the final modelling dataframe (only causal variables)
model_cols = ['income_segment', 'price_tier', 'purchase_freq',
              'basket_variety', 'dominant_season', 'high_demand']

model_df = causal_df[model_cols].copy()

print("✅ Final modelling dataframe ready")
print(f"   Shape: {model_df.shape}")
print()
print("--- Preview ---")
print(model_df.head(10))
print()
print("--- All columns are string/categorical: ---")
print(model_df.dtypes)

---
## Step 5: Structure Learning — Letting the Data Suggest a Causal Graph

This is the most important and most interesting step.

### What is a Causal Graph (DAG)?
A **Directed Acyclic Graph (DAG)** is a map of causal relationships:
- Each **node** is a variable (e.g., `income_segment`, `price_tier`)
- Each **arrow** (directed edge) means "this variable causally influences that one"
- **Acyclic** means no circular loops (income → demand → income would be a loop — not allowed)

### NOTEARS Algorithm
CausalNex uses the **NOTEARS** algorithm to learn a DAG from data. It finds the graph structure that best explains the statistical dependencies in the data, subject to the acyclicity constraint.

**Key parameter: `w_threshold`**  
Controls how many edges to keep. Higher threshold = fewer, stronger edges. We start with `0.2` and then prune implausible edges using domain knowledge.

> ⚠️ **Important:** NOTEARS works best with numeric data. We encode our categories as ordinal integers for structure learning, then revert to strings for the Bayesian Network fitting.

In [ ]:
# Step 5a: Encode categories as ordinal integers for NOTEARS structure learning
print("⚙️  Encoding categorical variables as ordinal integers for NOTEARS...")
print()

# Define ordinal mappings (order matters for causal interpretation)
ordinal_maps = {
    'income_segment':  {'Low_Income': 0, 'Mid_Income': 1, 'High_Income': 2},
    'price_tier':      {'Budget': 0, 'Mid_Range': 1, 'Premium': 2},
    'purchase_freq':   {'Infrequent': 0, 'Regular': 1, 'Frequent': 2},
    'basket_variety':  {'Narrow_Basket': 0, 'Broad_Basket': 1},
    'dominant_season': {'Spring': 0, 'Summer': 1, 'Autumn': 2, 'Winter': 3},
    'high_demand':     {'Low_Demand': 0, 'High_Demand': 1},
}

numeric_df = model_df.copy()
for col, mapping in ordinal_maps.items():
    numeric_df[col] = numeric_df[col].map(mapping)
    print(f"   {col}: {mapping}")

print()
print("--- Numeric encoded dataframe (first 5 rows) ---")
print(numeric_df.head())
print()
print("--- Any nulls after encoding? ---")
print(numeric_df.isnull().sum())

In [ ]:
# Step 5b: Run NOTEARS structure learning
print("⚙️  Running NOTEARS structure learning algorithm...")
print("   (This learns the causal graph from data)")
print()

sm = from_pandas(numeric_df, w_threshold=0.2)

print(f"✅ Structure learning complete")
print()
print(f"--- Discovered edges (before pruning) ---")
print(f"   Total edges found: {len(sm.edges)}")
print()
for edge in sorted(sm.edges(data=True), key=lambda x: abs(x[2].get('weight', 0)), reverse=True):
    source, target, data = edge
    weight = data.get('weight', 0)
    print(f"   {source}  →  {target}   (weight: {weight:.4f})")

---
## Step 6: Domain-Knowledge Pruning

NOTEARS discovers edges that are **statistically plausible**. But not all of them are **causally plausible** from a business standpoint.

This is where **your domain knowledge** enters the process. You review each edge and ask:

> *"Does it make real-world sense that A causes B — or could B cause A, or is there no real causal link?"*

### Our Pruning Decisions

| Edge | Decision | Reason |
|---|---|---|
| `income_segment → price_tier` | ✅ Keep | Higher income customers tend to buy premium products |
| `income_segment → high_demand` | ✅ Keep | Income directly affects how much people buy |
| `price_tier → high_demand` | ✅ Keep | Price level affects purchase quantity |
| `income_segment → purchase_freq` | ✅ Keep | Wealthier customers shop more often |
| `purchase_freq → high_demand` | ✅ Keep | More trips = more total volume |
| `dominant_season → high_demand` | ✅ Keep | Seasonality drives demand (Q4 is huge in retail) |
| `basket_variety → high_demand` | ✅ Keep | Customers who explore more products tend to buy more |
| `high_demand → income_segment` | ❌ Remove | Demand can't cause income (reverse causation) |
| `price_tier → income_segment` | ❌ Remove | Price paid doesn't cause income |

> **Rule of thumb:** If B existed before A in time, or if B is a structural variable (like income), it cannot be caused by a behavioural outcome variable like demand.

In [ ]:
# Step 6a: Remove causally implausible edges
print("⚙️  Pruning edges based on domain knowledge...")
print()

edges_to_remove = [
    # Reverse causation: demand can't cause income or price tier
    ('high_demand', 'income_segment'),
    ('high_demand', 'price_tier'),
    ('high_demand', 'purchase_freq'),
    ('high_demand', 'dominant_season'),
    # Price paid can't cause income
    ('price_tier', 'income_segment'),
    # Season is exogenous — nothing causes it
    ('income_segment', 'dominant_season'),
    ('price_tier', 'dominant_season'),
    ('purchase_freq', 'dominant_season'),
]

removed_count = 0
for edge in edges_to_remove:
    if sm.has_edge(*edge):
        sm.remove_edge(*edge)
        print(f"   ❌ Removed: {edge[0]} → {edge[1]}")
        removed_count += 1
    else:
        print(f"   ⚪ Not present (skipping): {edge[0]} → {edge[1]}")

print()
print(f"   Total edges removed: {removed_count}")
print(f"   Remaining edges: {len(sm.edges)}")

In [ ]:
# Step 6b: Add edges we KNOW should exist based on domain knowledge
# (NOTEARS may have missed some with low weight)
print("⚙️  Adding domain-knowledge edges that should exist...")
print()

edges_to_add = [
    ('income_segment', 'price_tier'),    # Income → price tier is fundamental
    ('income_segment', 'high_demand'),   # Income → demand is fundamental
    ('price_tier', 'high_demand'),       # Price → demand is fundamental
    ('dominant_season', 'high_demand'),  # Seasonality → demand is fundamental
    ('purchase_freq', 'high_demand'),    # Frequency → volume is logical
    ('basket_variety', 'high_demand'),   # Variety seeking → higher volume
    ('income_segment', 'purchase_freq'), # Income → shopping frequency
]

for edge in edges_to_add:
    if not sm.has_edge(*edge):
        sm.add_edge(*edge)
        print(f"   ✅ Added: {edge[0]} → {edge[1]}")
    else:
        print(f"   ✅ Already present: {edge[0]} → {edge[1]}")

print()
print(f"--- Final causal graph edges ---")
for edge in sm.edges():
    print(f"   {edge[0]}  →  {edge[1]}")

print()
print(f"Total edges in final graph: {len(sm.edges)}")

In [ ]:
# Step 6c: Visualise the causal graph
print("⚙️  Plotting causal graph...")

plt.figure(figsize=(14, 9))
plt.title("Causal Graph: Demand, Price & Income", fontsize=16, fontweight='bold', pad=20)

# Node positions (manually set for clarity)
pos = {
    'income_segment':  (0.0, 0.8),
    'dominant_season': (0.5, 1.0),
    'price_tier':      (0.2, 0.5),
    'purchase_freq':   (0.8, 0.5),
    'basket_variety':  (0.5, 0.3),
    'high_demand':     (0.5, 0.0),
}

# Color nodes by type
node_colors = {
    'income_segment':  '#4A90D9',  # blue  = structural/demographic
    'dominant_season': '#7B68EE',  # purple = exogenous
    'price_tier':      '#F5A623',  # orange = pricing
    'purchase_freq':   '#50C878',  # green  = behaviour
    'basket_variety':  '#50C878',  # green  = behaviour
    'high_demand':     '#E74C3C',  # red    = outcome
}

colors = [node_colors.get(n, '#AAAAAA') for n in sm.nodes()]

nx.draw_networkx(
    sm, pos=pos,
    node_color=colors,
    node_size=3000,
    font_size=9,
    font_color='white',
    font_weight='bold',
    arrows=True,
    arrowsize=25,
    edge_color='#555555',
    width=2.0,
    connectionstyle='arc3,rad=0.1'
)

# Legend
from matplotlib.patches import Patch
legend = [
    Patch(color='#4A90D9', label='Structural (Income)'),
    Patch(color='#7B68EE', label='Exogenous (Season)'),
    Patch(color='#F5A623', label='Pricing'),
    Patch(color='#50C878', label='Behaviour'),
    Patch(color='#E74C3C', label='Outcome (Demand)'),
]
plt.legend(handles=legend, loc='lower left', fontsize=9)

plt.axis('off')
plt.tight_layout()
plt.savefig('causal_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Causal graph saved as causal_graph.png")

---
## Step 7: Fit the Bayesian Network

Now that we have a causal graph (the *structure*), we fit a **Bayesian Network** to learn the conditional probability tables (CPTs) from data.

### What is a Conditional Probability Table?
For each node, the CPT tells us:
> *"Given the values of this node's parents in the graph, what is the probability distribution over this node's states?"*

For example, for `high_demand`:
> *"Given that income = High_Income AND price_tier = Budget, what is P(high_demand = High_Demand)?"*

**Bayes Lasso** is used during fitting to prevent overfitting on small data — it shrinks unreliable probability estimates toward neutral values.

In [ ]:
# Step 7a: Fit the Bayesian Network on the STRING (categorical) dataframe
print("⚙️  Fitting Bayesian Network to data...")
print()

bn = BayesianNetwork(sm)

# Fit node states (what values each variable can take)
bn = bn.fit_node_states(model_df)

print("--- Node states (all possible values per variable) ---")
for node, states in bn.node_states.items():
    print(f"   {node}: {sorted(states)}")

print()

In [ ]:
# Step 7b: Fit conditional probability distributions (CPDs)
print("⚙️  Fitting conditional probability distributions (CPDs)...")
print()

bn = bn.fit_cpds(model_df, method="BayesianEstimator", bayes_prior="K2")

print("✅ Bayesian Network fitted successfully")
print()

# Print the CPD for our key outcome: high_demand
print("--- CPD for high_demand (our outcome variable) ---")
print(bn.cpds['high_demand'])
print()

In [ ]:
# Step 7c: Print CPD for income_segment (root node — no parents)
print("--- CPD for income_segment (root node, no parents) ---")
print(bn.cpds['income_segment'])
print()

# And for price_tier (parent = income_segment)
print("--- CPD for price_tier (parent = income_segment) ---")
print(bn.cpds['price_tier'])
print()

---
## Step 8: Inference — Querying the Model

Now the real power begins. We use the **InferenceEngine** to:

1. **Query marginals** — *"In general, what % of customers show High_Demand?"*
2. **Observe evidence** — *"Given that we observe a customer has High_Income, what happens to P(High_Demand)?"* (this is like updating a belief with new info)
3. **Apply interventions (do-operator)** — *"If we **force** all customers to buy Budget-tier products, what happens to demand?"* (this is causal, not just correlational)

### Key Distinction: Observe vs. Intervene

| Method | Meaning | Business Interpretation |
|---|---|---|
| `query({'price_tier': 'Budget'})` | "Among customers we *observe* buying Budget" | Descriptive — who buys cheap |
| `do_intervention('price_tier', {'Budget': 1.0})` | "If we *force* all prices to Budget level" | Causal — what happens if we act |

In [ ]:
# Step 8a: Set up inference engine and check baseline
print("⚙️  Setting up Inference Engine...")
ie = InferenceEngine(bn)

# Baseline marginals — no evidence, no intervention
baseline = ie.query()

print("\n=" * 50)
print("BASELINE (no conditions applied)")
print("=" * 50)
print()
for node, probs in baseline.items():
    print(f"  📊 {node}:")
    for state, prob in sorted(probs.items()):
        bar = '█' * int(prob * 30)
        print(f"      {state:<20} {prob:.3f}  {bar}")
    print()

In [ ]:
# Step 8b: OBSERVATION — what do we see for high-income customers?
# This updates probabilities based on observing income = High_Income
# (correlational, not causal)

print("=" * 50)
print("OBSERVATION: What happens when we observe High_Income customers?")
print("=" * 50)
print()

baseline_demand     = baseline['high_demand']
high_income_obs     = ie.query({'income_segment': 'High_Income'})
low_income_obs      = ie.query({'income_segment': 'Low_Income'})

print("  high_demand given income = Low_Income:")
for state, prob in sorted(low_income_obs['high_demand'].items()):
    print(f"      {state:<20} {prob:.3f}")
print()

print("  high_demand given income = High_Income:")
for state, prob in sorted(high_income_obs['high_demand'].items()):
    print(f"      {state:<20} {prob:.3f}")
print()

lift = high_income_obs['high_demand']['High_Demand'] - low_income_obs['high_demand']['High_Demand']
print(f"  💡 Observation-based lift (High vs Low income): +{lift:.3f} ({lift*100:.1f} percentage points)")
print("     NOTE: This is correlational. High-income customers also buy premium products, etc.")

In [ ]:
# Step 8c: OBSERVATION — effect of price tier on demand
print()
print("=" * 50)
print("OBSERVATION: Price tier vs demand (correlational)")
print("=" * 50)
print()

for tier in ['Budget', 'Mid_Range', 'Premium']:
    q = ie.query({'price_tier': tier})
    p_high = q['high_demand']['High_Demand']
    bar = '█' * int(p_high * 40)
    print(f"  price_tier = {tier:<12} → P(High_Demand) = {p_high:.3f}  {bar}")

print()
print("  💡 Note: This is OBSERVATION, not intervention.")
print("     Budget-tier buyers may have lower demand because they are also low-income.")
print("     We need the do-operator to isolate the pure price effect.")

In [ ]:
# Step 8d: INTERVENTION (do-operator) — force price tier to Budget for ALL customers
# This is the causal question: "If we CUT prices to Budget level, what happens to demand?"
# The do-operator severs the edge income_segment → price_tier
# (removes the confounding: rich people choosing premium is no longer a factor)

print()
print("=" * 50)
print("INTERVENTION (do-operator): Forcing price_tier = Budget for ALL customers")
print("Business question: If we cut all prices to the Budget tier, what happens to demand?")
print("=" * 50)
print()

# Baseline demand before intervention
p_high_baseline = baseline['high_demand']['High_Demand']
print(f"  Baseline P(High_Demand):           {p_high_baseline:.4f}  ({p_high_baseline*100:.1f}%)")

# Intervene: force Budget price for everyone
ie.do_intervention('price_tier', {'Budget': 1.0, 'Mid_Range': 0.0, 'Premium': 0.0})
after_budget = ie.query()
p_high_budget = after_budget['high_demand']['High_Demand']
print(f"  After do(price_tier=Budget):       {p_high_budget:.4f}  ({p_high_budget*100:.1f}%)")
ie.reset_do('price_tier')

# Intervene: force Premium price for everyone
ie.do_intervention('price_tier', {'Budget': 0.0, 'Mid_Range': 0.0, 'Premium': 1.0})
after_premium = ie.query()
p_high_premium = after_premium['high_demand']['High_Demand']
print(f"  After do(price_tier=Premium):      {p_high_premium:.4f}  ({p_high_premium*100:.1f}%)")
ie.reset_do('price_tier')

print()
budget_lift  = p_high_budget  - p_high_baseline
premium_drop = p_high_premium - p_high_baseline
print(f"  📈 Causal effect of forcing Budget prices:  {budget_lift:+.4f} ({budget_lift*100:+.1f}pp)")
print(f"  📉 Causal effect of forcing Premium prices: {premium_drop:+.4f} ({premium_drop*100:+.1f}pp)")
print()
print("  💡 THIS is the actionable insight — not correlation.")
print("     The do-operator severs the income→price confounding path,")
print("     so we see the PURE effect of a price change on demand.")

In [ ]:
# Step 8e: INTERVENTION — what if all customers aspired to high purchase frequency?
# E.g., loyalty programme that forces frequent purchasing behaviour

print()
print("=" * 50)
print("INTERVENTION: What if we force all customers to be Frequent buyers?")
print("(e.g., a subscription or loyalty programme)")
print("=" * 50)
print()

p_high_baseline = baseline['high_demand']['High_Demand']
print(f"  Baseline P(High_Demand):                    {p_high_baseline:.4f}  ({p_high_baseline*100:.1f}%)")

ie.do_intervention('purchase_freq', {'Infrequent': 0.0, 'Regular': 0.0, 'Frequent': 1.0})
after_freq = ie.query()
p_high_freq = after_freq['high_demand']['High_Demand']
ie.reset_do('purchase_freq')

ie.do_intervention('purchase_freq', {'Infrequent': 1.0, 'Regular': 0.0, 'Frequent': 0.0})
after_infreq = ie.query()
p_high_infreq = after_infreq['high_demand']['High_Demand']
ie.reset_do('purchase_freq')

print(f"  After do(purchase_freq=Infrequent):         {p_high_infreq:.4f}  ({p_high_infreq*100:.1f}%)")
print(f"  After do(purchase_freq=Frequent):           {p_high_freq:.4f}  ({p_high_freq*100:.1f}%)")
print()
print(f"  📈 Causal lift from loyalty programme: {p_high_freq - p_high_infreq:+.4f} ({(p_high_freq - p_high_infreq)*100:+.1f}pp)")

In [ ]:
# Step 8f: Combined intervention — Budget price AND Frequent buyers
# E.g., "What if we ran a deep discount loyalty programme?"

print()
print("=" * 50)
print("COMBINED INTERVENTION: Budget price + Frequent buyers")
print("Business: Deep-discount loyalty programme — does it compound?")
print("=" * 50)
print()

ie.do_intervention('price_tier', {'Budget': 1.0, 'Mid_Range': 0.0, 'Premium': 0.0})
ie.do_intervention('purchase_freq', {'Infrequent': 0.0, 'Regular': 0.0, 'Frequent': 1.0})
combined = ie.query()
p_combined = combined['high_demand']['High_Demand']
ie.reset_do('price_tier')
ie.reset_do('purchase_freq')

print(f"  Baseline:                   {p_high_baseline:.4f}  ({p_high_baseline*100:.1f}%)")
print(f"  Budget only:                {p_high_budget:.4f}  ({p_high_budget*100:.1f}%)")
print(f"  Frequent only:              {p_high_freq:.4f}  ({p_high_freq*100:.1f}%)")
print(f"  Budget + Frequent combined: {p_combined:.4f}  ({p_combined*100:.1f}%)")
print()
print(f"  💡 Combined causal lift: {p_combined - p_high_baseline:+.4f} ({(p_combined - p_high_baseline)*100:+.1f}pp)")

---
## Step 9: Summary Visualisation — Causal Effects Dashboard

Let's put all the intervention results together in a clear chart that a business stakeholder could look at.

In [ ]:
# Step 9: Summary bar chart of causal interventions

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Causal Inference Results: What Actually Drives High Demand?', 
             fontsize=15, fontweight='bold', y=1.02)

# --- Left chart: Observation vs Intervention for price tier ---
ax1 = axes[0]

obs_results = {}
do_results  = {}

for tier in ['Budget', 'Mid_Range', 'Premium']:
    # Observation
    q = ie.query({'price_tier': tier})
    obs_results[tier] = q['high_demand']['High_Demand']
    # Intervention
    ie.do_intervention('price_tier', {t: (1.0 if t == tier else 0.0) for t in ['Budget','Mid_Range','Premium']})
    q2 = ie.query()
    do_results[tier] = q2['high_demand']['High_Demand']
    ie.reset_do('price_tier')

x = np.arange(3)
labels = list(obs_results.keys())
obs_vals = [obs_results[l] for l in labels]
do_vals  = [do_results[l]  for l in labels]

bars1 = ax1.bar(x - 0.2, obs_vals, 0.35, label='Observation (correlational)', color='#AED6F1', edgecolor='black')
bars2 = ax1.bar(x + 0.2, do_vals,  0.35, label='Intervention / do() (causal)', color='#2E86C1', edgecolor='black')

ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=11)
ax1.set_ylabel('P(High Demand)', fontsize=11)
ax1.set_title('Price Tier: Observe vs Intervene', fontsize=12, fontweight='bold')
ax1.axhline(y=p_high_baseline, color='red', linestyle='--', linewidth=1.5, label=f'Baseline ({p_high_baseline:.2f})')
ax1.set_ylim(0, 1)
ax1.legend(fontsize=9)

for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# --- Right chart: All intervention effects ranked ---
ax2 = axes[1]

interventions = {
    'Price = Budget\n(price cut)':        p_high_budget - p_high_baseline,
    'Price = Premium\n(price hike)':      p_high_premium - p_high_baseline,
    'Loyalty (Frequent)':                  p_high_freq - p_high_baseline,
    'Lapsed (Infrequent)':                 p_high_infreq - p_high_baseline,
    'Budget + Loyal\n(combined)':          p_combined - p_high_baseline,
}

sorted_items = sorted(interventions.items(), key=lambda x: x[1])
names  = [i[0] for i in sorted_items]
values = [i[1] for i in sorted_items]
colors_bar = ['#E74C3C' if v < 0 else '#27AE60' for v in values]

bars = ax2.barh(names, values, color=colors_bar, edgecolor='black', height=0.5)
ax2.axvline(x=0, color='black', linewidth=1)
ax2.set_xlabel('Change in P(High Demand) vs Baseline', fontsize=11)
ax2.set_title('Causal Effect of Each Intervention\n(vs Baseline)', fontsize=12, fontweight='bold')

for bar, val in zip(bars, values):
    ax2.text(val + (0.003 if val >= 0 else -0.003),
             bar.get_y() + bar.get_height()/2,
             f'{val:+.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('causal_results_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved as causal_results_dashboard.png")

---
## Step 10: Business Interpretation

Let's translate the causal results into plain English for a business stakeholder.

In [ ]:
# Step 10: Final business summary

print("=" * 60)
print("BUSINESS SUMMARY: Causal Drivers of High Demand")
print("=" * 60)
print()
print(f"  Baseline P(High Demand) = {p_high_baseline*100:.1f}%")
print()
print("  ─── Price Interventions ───")
print(f"  If we force ALL prices to Budget tier:")
print(f"    P(High Demand) → {p_high_budget*100:.1f}%  (change: {(p_high_budget-p_high_baseline)*100:+.1f}pp)")
print()
print(f"  If we force ALL prices to Premium tier:")
print(f"    P(High Demand) → {p_high_premium*100:.1f}%  (change: {(p_high_premium-p_high_baseline)*100:+.1f}pp)")
print()
print("  ─── Loyalty / Frequency Interventions ───")
print(f"  If all customers become Frequent buyers (loyalty programme):")
print(f"    P(High Demand) → {p_high_freq*100:.1f}%  (change: {(p_high_freq-p_high_baseline)*100:+.1f}pp)")
print()
print(f"  If all customers become Infrequent (no retention):")
print(f"    P(High Demand) → {p_high_infreq*100:.1f}%  (change: {(p_high_infreq-p_high_baseline)*100:+.1f}pp)")
print()
print("  ─── Combined Strategy ───")
print(f"  Budget pricing + Loyalty programme combined:")
print(f"    P(High Demand) → {p_combined*100:.1f}%  (change: {(p_combined-p_high_baseline)*100:+.1f}pp)")
print()
print("=" * 60)
print("  KEY TAKEAWAYS")
print("=" * 60)
print()
print("  1. Price cuts and loyalty programmes both causally lift demand")
print("     — but the magnitude tells you WHICH lever is more valuable.")
print()
print("  2. The do-operator result is DIFFERENT from simple observation.")
print("     Observing Budget customers ≠ forcing everyone to Budget prices.")
print("     The gap between these two numbers is the confounding bias.")
print()
print("  3. Combined interventions may not compound linearly.")
print("     If Budget + Loyal < Budget_effect + Loyal_effect,")
print("     the levers share a causal pathway (through purchase_freq).")
print()
print("  4. Next step: Validate with an A/B test before acting.")
print("     Causal inference narrows WHERE to experiment; it doesn't")
print("     replace experimentation entirely.")
print("=" * 60)

---
## 📚 What to Learn Next

Now that you've run end-to-end causal inference with CausalNex, here are the natural next steps:

| Topic | What it adds |
|---|---|
| **Cross-validation of structure** | Use `BayesianNetwork.fit_cpds` with train/test splits to check stability |
| **NOTEARS with constraints** | Force certain edges to exist/not exist via `tabu_edges` |
| **DoWhy library** | More formal causal identification, refutation tests |
| **Propensity score matching** | Alternative to Bayesian Networks for ATE estimation |
| **A/B test design using CI** | Use causal graph to decide *which* variable to intervene on |
| **Sensitivity analysis** | How much do results change if one edge is wrong? |

---
### Key Vocabulary Recap

| Term | Plain English |
|---|---|
| DAG | A map of cause-and-effect arrows between variables |
| Bayesian Network | A DAG with probability tables attached to each node |
| NOTEARS | Algorithm that learns the DAG structure from data |
| do-operator | Forces a variable to a specific value — simulates a real intervention |
| Confounding | A third variable that causes both X and Y, making X→Y look causal when it isn't |
| CPD | Conditional Probability Distribution — P(node | parents) |
| ATE | Average Treatment Effect — expected causal lift across the whole population |